In [15]:
import pandas as pd

train_essays = pd.read_parquet('/home/jack/github/kaggle/scoring/train_essays.parquet')
val_essays = pd.read_parquet('/home/jack/github/kaggle/scoring/validation_essays.parquet')

# train_essays = train_essays.sample(500).reset_index(drop=True)
# validation_essays = validation_essays.sample(500).reset_index(drop=True)

In [16]:
import pandas as pd
from sklearn.utils import resample

def sample_df(df, col='score', sub=0, random_state=None):
    """
    Balances the classes in a DataFrame by resampling.
    
    Parameters:
    - df: DataFrame to be resampled.
    - col: The column name in df that contains class labels.
    - random_state: The random state for reproducibility.
    
    Returns:
    - balanced_df: A DataFrame with balanced classes.
    """
    class_counts = df[col].value_counts()
    target_count = max(int(class_counts.median()) - sub, class_counts.min())  # Ensure target_count is positive
    
    balanced_df = pd.DataFrame()

    for class_label in df[col].unique():
        class_subset = df[df[col] == class_label]
        
        if len(class_subset) > target_count:
            # Downsample majority classes
            class_subset = resample(class_subset,
                                    replace=False,
                                    n_samples=target_count,
                                    random_state=random_state)
        else:
            # Upsample minority classes
            class_subset = resample(class_subset,
                                    replace=True,
                                    n_samples=target_count,
                                    random_state=random_state)
        
        balanced_df = pd.concat([balanced_df, class_subset], axis=0)
    
    # Shuffle the DataFrame to mix the classes well
    balanced_df = balanced_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return balanced_df


In [17]:
# Balance only the training DataFrame
sampled_train_essays = sample_df(train_essays, col='score', random_state=42, sub=0)

sampled_val_essays = sample_df(val_essays, col='score', sub=0, random_state=42)

In [18]:
STATUS = 'Post'

if STATUS == 'Post':
    
    drop_cols = [ 'full_text', 'lowered', 'clean_text','corrected_text', 'segmented_text']

# else:
    
#     drop_cols = ['full_text', 'lowered', 'clean_text',
#         'Pre_tokens', 'Pre_sentences', 'Pre_pos_tags',
#         'corrected_text', 'segmented_text',]



train_df = sampled_train_essays.copy()
val_df = sampled_val_essays.copy()

train_df.drop(columns=drop_cols, inplace= True)
val_df.drop(columns=drop_cols, inplace= True)

In [19]:
feature_cols = []

for col in train_df.columns:
    if (col != 'essay_id') and (col != 'score'):
        feature_cols.append(col) 

In [20]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import pickle
import numpy as np

# Assuming 'feature_cols' are defined elsewhere in your script

# Labels (stays the same)
sampled_train_labels = np.array(train_df['score'])
sampled_val_labels = np.array(val_df['score'])
full_train_labels = np.array(train_essays['score'])
full_val_labels = np.array(val_essays['score'])

# sampled_train_labels = pd.DataFrame(sampled_train_labels, columns=['score'],
#                              index=sampled_train_labels.index)

# sampled_val_labels = pd.DataFrame(sampled_val_labels, columns=['score'], 
#                           index=sampled_val_labels.index)


# full_train_labels = pd.DataFrame(full_train_labels, columns=['score'],
#                              index=full_train_labels.index)

# full_val_labels = pd.DataFrame(full_val_labels, columns=['score'], 
#                           index=full_val_labels.index)



# Features
sampled_train_features = train_df[feature_cols]
sampled_val_features = val_df[feature_cols]
full_train_features = train_essays[feature_cols]
full_val_features = val_essays[feature_cols]

# Initialize scaler
scaler = StandardScaler()

# Fit scaler to the original full training data
scaler.fit(full_train_features)

# Transform all datasets using the fitted scaler
sampled_train_feats_scaled = scaler.transform(sampled_train_features)
sampled_val_feats_scaled = scaler.transform(sampled_val_features)
full_train_feats_scaled = scaler.transform(full_train_features)
full_val_feats_scaled = scaler.transform(full_val_features)

# Save the scaler for later use
with open('sklearn_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)


In [21]:
from sklearn.metrics import make_scorer, cohen_kappa_score

def quadratic_weighted_kappa_scorer(y_true, y_pred):
    """
    Compute the Quadratic Weighted Kappa (QWK), also known as Cohen's kappa.
    
    Parameters:
    y_true : array-like of shape (n_samples,)
        True labels.
    y_pred : array-liimport keras_tuner
from sklearn import ensemble
from sklearn import datasets
from sklearn import linear_model
from sklearn import metrics
from sklearn import model_selection

def build_model(hp):
  model_type = hp.Choice('model_type', ['random_forest', 'ridge'])
  if model_type == 'random_forest':
    model = ensemble.RandomForestClassifier(
        n_estimators=hp.Int('n_estimators', 10, 50, step=10),
        max_depth=hp.Int('max_depth', 3, 10))
  else:
    model = linear_model.RidgeClassifier(
        alpha=hp.Float('alpha', 1e-3, 1, sampling='log'))
  return model

tuner = keras_tuner.tuners.SklearnTuner(
    oracle=keras_tuner.oracles.BayesianOptimizationOracle(
        objective=keras_tuner.Objective('score', 'max'),
        max_trials=100),
    hypermodel=build_model,
    scoring=metrics.make_scorer(metrics.accuracy_score),
    cv=model_selection.StratifiedKFold(10),
    directory='.',
    project_name='my_project')ke of shape (n_samples,)
        Predicted labels.
    
    Returns:
    score : float
        Quadratic Weighted Kappa score.
    """
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

In [22]:
import lazypredict
from lazypredict.Supervised import LazyClassifier
from sklearn.model_selection import train_test_split
import pandas as pd

# Assuming df is your DataFrame and 'target' is your target column
X_train = full_train_feats_scaled
y_train = full_train_labels

X_test = full_val_feats_scaled
y_test = full_val_labels


# Create and fit the LazyClassifier

clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric= None)    #quadratic_weighted_kappa_scorer)
models, predictions = clf.fit(X_train, X_test, y_train, y_test)

# Display the performance metrics of the models
print(models)


 97%|█████████▋| 28/29 [2:33:07<06:41, 401.93s/it]   

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.863840 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3496114
[LightGBM] [Info] Number of data points in the train set: 12980, number of used features: 13821
[LightGBM] [Info] Start training from score -2.623160
[LightGBM] [Info] Start training from score -1.305517
[LightGBM] [Info] Start training from score -1.001693
[LightGBM] [Info] Start training from score -1.488407
[LightGBM] [Info] Start training from score -2.901684
[LightGBM] [Info] Start training from score -4.770685


100%|██████████| 29/29 [2:38:04<00:00, 327.06s/it]

                               Accuracy  Balanced Accuracy ROC AUC  F1 Score  \
Model                                                                          
LGBMClassifier                     0.57               0.40    None      0.55   
BaggingClassifier                  0.54               0.37    None      0.52   
DecisionTreeClassifier             0.45               0.34    None      0.44   
AdaBoostClassifier                 0.34               0.31    None      0.28   
BernoulliNB                        0.29               0.31    None      0.30   
RandomForestClassifier             0.52               0.30    None      0.48   
NearestCentroid                    0.25               0.29    None      0.26   
LogisticRegression                 0.40               0.27    None      0.39   
GaussianNB                         0.23               0.27    None      0.24   
SGDClassifier                      0.42               0.27    None      0.40   
ExtraTreesClassifier               0.47 

In [9]:
import sklearn
print(sklearn.__version__)


1.4.2


In [10]:
# pandas display all the colmns in pd.columns
pd.set_option('display.max_columns', None)
